# NeuroLogit quickstart: an sklearn-style logistic model for neuroscience applications

When modelling behaviour in perceptual decision-making tasks, logistic classification is [often useful](https://www.sciencedirect.com/science/article/pii/S0896627324004513?via%3Dihub). However, you often need to scale your input non-linearly (a psychophysical scaling problem going back to Fechner), or assess the contribution of individual parameters to your data — neither of which is straightforward in something like scikit-learn.

This package solves both problems with a set of [purpose-built classes](https://github.com/takacsflora/NeuroLogit/blob/main/NeuroLogit/src/models/model_base_scipy.py). There are two useful classes, `Logit` and `MultinomialLogit`, both built on `BaseTrainer` — a skeleton for writing scipy-based models that support fixing parameters and imposing bounds on them.

[This example](https://github.com/takacsflora/NeuroLogit/blob/main/NeuroLogit/src/models/av_models_opto.py#L69) shows how to use the `Logit` class effectively; basically all you need to do is to: 
1) Define your parameters in the `__init__` function. Define your parameter variables as a list, with their initial values and bounds as dictinaries. For bounds the lower and the upper bound needs to be encoded as a tuple. 
2) define the rule that predicts the odds based on your parameters in the `predict_log_proba` funtion

These modules work using class nheritance, you can read about that[here](https://realpython.com/inheritance-composition-python/). Unless you specify the init and the bounds they will be 1 and (None,None)


In [ ]:
# when defining your model you need to make sure you initialised it with the parameters you want

from NeuroLogit.src.models.model_base_scipy import Logit

class my_awesome_model(Logit):
    def __init__(self):
        param_names = ['vis_slope','gamma','bias']
        param_init = {
            'bias': 0,
        }

        param_bounds = {
            "gamma": (0.2, 2),
        }
        
        super().__init__(param_names=param_names, param_init=param_init, param_bounds=param_bounds)

    def predict_log_proba(self, X):
        # this is where you define your model, in this case we are using a simple linear model with a bias and a slope
        # the gamma parameter is used to scale the output of the model, it can be used to make the model more or less confident in its predictions
        # the bias parameter is used to shift the output of the model, it can be used to make the model more or less biased towards one class or another
        # the slope parameter is used to scale the input features, it can be used to make the model more or less sensitive to changes in the input features

        vis_slope = self.params['vis_slope']
        gamma = self.params['gamma']
        bias = self.params['bias']

        contrast_scaled = X ** gamma
        
        return contrast_scaled @ vis_slope + bias


You can then then fit your model on some aribtrarily generated cohice data. The Logit model class will predict two choice options 0 and 1, whose log-probabilies will negative and positive respectively. So make sure to encode your predicted choice classes as 0s and 1s. The main thig that this package was designed to do is that you can then fix your parameters, fit reduced models etc. essential for parameter testing! Here is the minimal working example:  


Another useful class is the `MultinomialLogit`. This class also allows nonlinearities and implements the fitting using a reference class (often used in econometrics). This can be useful if you are modelling left, right and NoGo choices, as NoGo is often the reference class. In this framework, the model estimates the log-odds of each class relative to the reference:

$$\ln\left(\frac{P(Right)}{P(NoGo)}\right) = f_{Right}(x)$$
$$\ln\left(\frac{P(Left)}{P(NoGo)}\right) = f_{Left}(x)$$

The probability for a specific class $i$ is then calculated as:

$$P(i) = \frac{e^{f_i(x)}}{1 + \sum_{j=1}^{K-1} e^{f_j(x)}}$$

You can see more of how I used this model in [my paper](https://www.biorxiv.org/content/10.64898/2026.06.05.730072v1). 

**Note on comparison with sklearn**: Fitting multinomial logit with a reference class is different from the multinomial logistic regression versions scikit-learn offers (symmetric softmax or one-vs-rest). Those versions fit a weight vector for every class, including the reference class, resulting in K weight vectors for K classes. In contrast, fitting with a reference class only requires K-1 vectors. While the symmetric model is useful for deep learning, it is over-parameterised, which limits its utility when you want to interpret the actual parameters. 

You can use `MultinomialLogit` pretty much the same way as `Logit`. The only difference is that now your `predict_log_proba` must output a tuple of left and right decision variables (`zL`,`zR`), and that the choices need to be encided as [0,1,2] values where 0 is the reference class. For example: 

In [ ]:
from NeuroLogit.src.models.model_base_scipy import MultinomialLogit

class my_awesome_3choice_model(MultinomialLogit):
    def __init__(self):
        param_names = ['vR', 'vL', 'gamma', 'biasR', 'biasL']
        param_init = {
            'biasR': 0,
            'biasL': 0,
        }

        param_bounds = {
            "gamma": (0.2, 2),
        }
        
        super().__init__(param_names=param_names, param_init=param_init, param_bounds=param_bounds)

    def predict_log_proba(self, X):

        vR = self.params['vR']
        vL = self.params['vL']
        gamma = self.params['gamma']
        biasR = self.params['biasR']
        biasL = self.params['biasL']

        visR_gamma_scaled = X['visR_contrast'] ** gamma
        visL_gamma_scaled = X['visL_contrast'] ** gamma

        zR = visR_gamma_scaled @ vR + biasR
        zL = visL_gamma_scaled @ vL + biasL

        return zL,zR